# BT4012 — Fraudulent Bitcoin Transaction Detection

**Student number: e1156454**

This notebook reproduces the final Kaggle submission end to end: loading the supplied
data, building the graph features, validating chronologically, and writing the
submission file. It imports nothing from the project's `src/` package, so it runs from a
fresh clone given only the four competition CSVs.

## Result

| Submission | Public LB |
|---|---|
| `hgb` on raw features only | 0.94351 |
| `hgb` on `prop34_cmp` (2/3/4-hop propagation) | 0.95937 |
| `hgb` on `ppr_cmp` (PageRank propagation, α = 0.15) | 0.96034 |
| `hgb` on `ppr_multi_cmp` (three decays) | 0.96057 |
| `hgb` on `ppr_fine_multi_cmp` (six decays) | 0.96066 |
| **rank blend of the four** | **0.96124** |

## What produced the improvement

The 165 supplied features already include statistics over each transaction's
**immediate** neighbours. Hand-built one-hop aggregates therefore added nothing, and
removing the one-hop propagation block measurably helped. The gain came from information
**further out in the payment graph**: two to four hops away, or mixed across all
distances with a PageRank decay.

The model never changed. Histogram gradient boosting with class weighting, at its
default settings, is used for every model here. Four tuning searches (two properly
nested), stacking, bagging, pseudo-labelling, positive-unlabelled learning,
covariate-shift weighting, similarity-based label transfer, a GraphSAGE network and four
other gradient-boosting libraries were all tested and none improved on it.

Runtime: roughly 30–45 minutes.

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import rankdata
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

SEED = 42
np.random.seed(SEED)
pd.set_option("display.width", 170)

# Folder holding train.csv, test.csv, txs_edgelist.csv and sample_submission.csv.
# Path("data") when the notebook runs from the repository root; Path("../data") from notebooks/.
DATA_DIR = Path("../data")
OUT_DIR = Path("submissions")
OUT_DIR.mkdir(exist_ok=True)

## 1. Load the data

`label` is blank for unlabelled transactions, so it is parsed explicitly: `1` illicit,
`0` licit, blank / `none` / `unknown` missing. Any other value raises an error rather
than being silently treated as unknown.

In [2]:
def parse_label(raw):
    text = str(raw).strip().lower()
    if text in {"", "none", "unknown", "nan"}:
        return np.nan
    if text in {"0", "0.0"}:
        return 0.0
    if text in {"1", "1.0"}:
        return 1.0
    raise ValueError(f"Unexpected label value: {raw!r}")


train = pd.read_csv(DATA_DIR / "train.csv", converters={"label": parse_label})
test = pd.read_csv(DATA_DIR / "test.csv")
edges = pd.read_csv(DATA_DIR / "txs_edgelist.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
train["label"] = train["label"].astype(float)
edges.columns = ["src", "dst"]

print(f"train {train.shape}  steps {train.time_step.min()}-{train.time_step.max()}")
print(f"test  {test.shape}  steps {test.time_step.min()}-{test.time_step.max()}")
print(f"edges {len(edges):,}")
print(f"labels: {int((train.label == 0).sum()):,} licit, {int((train.label == 1).sum()):,} illicit, "
      f"{int(train.label.isna().sum()):,} unknown ({train.label.mean():.2%} illicit among labelled)")

train (141772, 168)  steps 1-35
test  (15329, 168)  steps 36-49
edges 234,355
labels: 27,591 licit, 3,644 illicit, 110,537 unknown (11.67% illicit among labelled)


## 2. Inspection and cleaning

The files need no cleaning: no missing or infinite feature values, no constant or
duplicated feature columns, no duplicate transaction IDs, no train/test ID overlap.
Three decisions follow from inspecting them.

**Drop six features.** `feat_100, 101, 103, 136, 137, 139` encode time almost perfectly:
train and test distributions barely overlap and their per-step means rise monotonically
with `time_step`. For three of them over 99% of test values exceed the training maximum,
which a tree cannot extrapolate to. The cell below prints that evidence.

**Never use `txId` or `time_step` as predictors.** `time_step` defines the splits only.

**Exclude unknown labels from fitting and scoring.** Those rows still exist in the graph,
subject to the coverage rule in section 3.

In [3]:
TIME_ENCODING = ["feat_100", "feat_101", "feat_103", "feat_136", "feat_137", "feat_139"]
RAW = [c for c in train.columns if c.startswith("feat_") and c not in TIME_ENCODING]

evidence = pd.DataFrame({
    "train_max": train[TIME_ENCODING].max(),
    "test_max": test[TIME_ENCODING].max(),
    "share_of_test_above_train_max": [(test[c] > train[c].max()).mean() for c in TIME_ENCODING],
})
print(evidence.round(3).to_string())
print(f"\nPredictors kept: {len(RAW)} of the 165 supplied features")

          train_max  test_max  share_of_test_above_train_max
feat_100      0.737     1.641                          0.992
feat_101      0.738     1.658                          0.995
feat_103      0.740     1.657                          0.994
feat_136      4.124     3.408                          0.000
feat_137      3.485     3.485                          0.000
feat_139      3.974     3.273                          0.000

Predictors kept: 159 of the 165 supplied features


## 3. Graph features

Three kinds of block are built from `txs_edgelist.csv`. None uses a label **value**.

**Coverage matching.** Test transactions are missing about 39% of their neighbours'
feature rows, because those transactions are absent from the files; training transactions
would otherwise have 100%. For the features to mean the same thing in both periods, only
*usable* rows contribute: labelled training rows and test rows. That gives 63% against
61% coverage instead of 100% against 61%. The alternatives — including unlabelled rows,
or relaying through featureless nodes — were tested and scored worse.

**Block A — components (`cmp_`).** Size of the weakly connected component a transaction
belongs to, the share of it with usable feature rows, and degree relative to component
size. Components use the complete edge list. Every edge lies within one time step, so
components never span periods.

**Block B — PageRank propagation.** `H ← (1−α)·P·H + α·X` over ten iterations, where `P`
is the row-normalised adjacency matrix: one block of columns mixing every hop distance
with a geometric decay. Six decays are built (α = 0.05, 0.10, 0.15, 0.20, 0.25, 0.30),
because mixing scales measurably beat any single scale on the leaderboard.

**Block C — discrete hops (`p2_`, `p3_`, `p4_`).** `P²X`, `P³X`, `P⁴X`. One-hop
propagation is deliberately **excluded**: the supplied features already encode one-hop
statistics, and including it cost 0.0016 on the leaderboard.

In [4]:
def graph_setup(train, test, edges, feature_cols):
    """Node table, feature matrix and the usable edge endpoints as row positions."""
    nodes = pd.concat([train[["txId", "time_step", "label"] + feature_cols],
                       test[["txId", "time_step"] + feature_cols].assign(label=1.0)],
                      ignore_index=True)
    usable = nodes["label"].notna().to_numpy()      # labelled train rows + all test rows
    pos = pd.Series(np.arange(len(nodes)), index=nodes["txId"].values)
    e = edges[edges["src"].isin(pos.index) & edges["dst"].isin(pos.index)]
    si, di = pos.loc[e["src"]].to_numpy(), pos.loc[e["dst"]].to_numpy()
    keep = usable[si] & usable[di]
    return nodes, nodes[feature_cols].to_numpy("float32"), si[keep], di[keep]


def transition_matrix(si, di, n):
    """Row-normalised adjacency: every row averages over its usable neighbours."""
    rows, cols = np.r_[si, di], np.r_[di, si]
    A = sparse.coo_matrix((np.ones(len(rows), "float32"), (rows, cols)), shape=(n, n)).tocsr()
    deg = np.asarray(A.sum(axis=1)).ravel()
    inv = np.zeros_like(deg)
    inv[deg > 0] = 1.0 / deg[deg > 0]
    return sparse.diags(inv.astype("float32")) @ A, deg


nodes, X, SI, DI = graph_setup(train, test, edges, RAW)
P, DEG = transition_matrix(SI, DI, len(nodes))
IDX = pd.Index(nodes["txId"].values, name="txId")
print(f"{len(nodes):,} nodes, {len(SI):,} usable edges, "
      f"{(DEG == 0).sum():,} nodes with no usable neighbour")

157,101 nodes, 36,624 usable edges, 121,227 nodes with no usable neighbour


In [5]:
# ── Block A: component features (complete edge list) ──────────────────────────
def component_features(edges, node_ids, usable_ids):
    all_ids = pd.unique(pd.concat([edges["src"], edges["dst"], pd.Series(node_ids)]))
    pos = pd.Series(np.arange(len(all_ids)), index=all_ids)
    parent = np.arange(len(all_ids))

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    for u, v in zip(pos.loc[edges["src"]].to_numpy(), pos.loc[edges["dst"]].to_numpy()):
        ru, rv = find(u), find(v)
        if ru != rv:
            parent[ru] = rv
    root = np.array([find(i) for i in range(len(all_ids))])

    df = pd.DataFrame({"root": root, "usable": pd.Index(all_ids).isin(usable_ids)})
    size = df.groupby("root")["usable"].size()
    share = df.groupby("root")["usable"].mean()
    degree = pd.concat([edges["src"], edges["dst"]]).value_counts()

    rows = pos.loc[node_ids].to_numpy()
    out = pd.DataFrame(index=pd.Index(node_ids, name="txId"))
    out["cmp_size"] = df["root"].map(size).to_numpy()[rows]
    out["cmp_usable_share"] = df["root"].map(share).to_numpy()[rows]
    out["cmp_degree_over_size"] = (pd.Index(node_ids).map(degree).fillna(0).to_numpy()
                                   / out["cmp_size"].to_numpy())
    return out.astype("float32")


usable_ids = set(nodes.loc[nodes["label"].notna(), "txId"])
cmp_feats = component_features(edges, nodes["txId"].to_numpy(), usable_ids)
print(cmp_feats.describe().round(2).to_string())

        cmp_size  cmp_usable_share  cmp_degree_over_size
count  157101.00         157101.00             157101.00
mean     4726.98              0.22                  0.00
std      1588.92              0.05                  0.00
min      1089.00              0.11                  0.00
25%      3537.00              0.18                  0.00
50%      4528.00              0.24                  0.00
75%      5894.00              0.27                  0.00
max      7880.00              0.43                  0.09


In [6]:
# ── Blocks B and C: PageRank decays and discrete hops ─────────────────────────
def ppr_features(X, P, deg, index, cols, alpha, prefix, iters=10):
    H = X.copy()
    for _ in range(iters):
        H = (1 - alpha) * (P @ H) + alpha * X
    out = pd.DataFrame(H, index=index, columns=[f"{prefix}{c}" for c in cols])
    out[deg == 0] = np.nan      # no usable neighbour: left missing, never filled
    return out.astype("float32")


def hop_features(X, P, deg, index, cols, hops=(2, 3, 4)):
    blocks, H = {}, X
    for hop in range(1, max(hops) + 1):
        H = P @ H
        if hop in hops:
            frame = pd.DataFrame(H, index=index, columns=[f"p{hop}_{c}" for c in cols])
            frame[deg == 0] = np.nan
            blocks[hop] = frame.astype("float32")
    return pd.concat(blocks.values(), axis=1)


ALPHAS = {"ppr_": 0.15, "ppr05_": 0.05, "ppr30_": 0.30,
          "pprf10_": 0.10, "pprf20_": 0.20, "pprf25_": 0.25}
ppr_blocks = {prefix: ppr_features(X, P, DEG, IDX, RAW, alpha=a, prefix=prefix)
              for prefix, a in ALPHAS.items()}
hops = hop_features(X, P, DEG, IDX, RAW)
print(f"{len(ppr_blocks)} PageRank blocks of {ppr_blocks['ppr_'].shape[1]} columns; "
      f"hop block {hops.shape[1]} columns")

6 PageRank blocks of 159 columns; hop block 477 columns


In [7]:
# ── Assemble the four feature sets ────────────────────────────────────────────
graph_block = pd.concat([cmp_feats] + list(ppr_blocks.values()) + [hops], axis=1)
train_f = train.join(graph_block, on="txId")
test_f = test.join(graph_block, on="txId")

CMP = list(cmp_feats.columns)
cols_of = lambda prefix: list(ppr_blocks[prefix].columns)
FEATURES = {
    "ppr_cmp":            RAW + cols_of("ppr_") + CMP,
    "prop34_cmp":         RAW + list(hops.columns) + CMP,
    "ppr_multi_cmp":      RAW + cols_of("ppr_") + cols_of("ppr05_") + cols_of("ppr30_") + CMP,
    "ppr_fine_multi_cmp": RAW + cols_of("ppr_") + cols_of("ppr05_") + cols_of("ppr30_")
                              + cols_of("pprf10_") + cols_of("pprf20_") + cols_of("pprf25_") + CMP,
}
for name, cols in FEATURES.items():
    print(f"{name:20s} {len(cols):5d} features")

ppr_cmp                321 features
prop34_cmp             639 features
ppr_multi_cmp          639 features
ppr_fine_multi_cmp    1116 features


## 4. Validation design

Test covers steps 36–49 while training covers 1–35, so validation must always predict
**later** periods from earlier ones. Random k-fold would leak future information and
overstate performance badly.

| Split | Train steps | Validation steps |
|---|---|---|
| fold1–fold4 | 1–15, 1–20, 1–25, 1–30 | the next 5 steps |
| horizon | 1–21 | 22–35 (14 steps, the length of the test horizon) |

Unknown labels are excluded from fitting and scoring. Every edge between known
transactions lies inside one time step, so a time-based split can never separate the two
ends of an edge.

In [8]:
SPLITS = [("fold1", (1, 15), (16, 20)), ("fold2", (1, 20), (21, 25)),
          ("fold3", (1, 25), (26, 30)), ("fold4", (1, 30), (31, 35)),
          ("horizon", (1, 21), (22, 35))]


def make_model():
    """The final model. Fixed settings and no early stopping, so the model retrained on
    all labelled rows is exactly the one that was validated."""
    return HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=300, max_leaf_nodes=31, min_samples_leaf=20,
        l2_regularization=0.0, early_stopping=False, class_weight="balanced",
        random_state=SEED)


def validate(frame, feature_cols, splits=SPLITS):
    labelled = frame[frame["label"].notna()]
    rows = []
    for name, (tr_lo, tr_hi), (va_lo, va_hi) in splits:
        tr = labelled[labelled["time_step"].between(tr_lo, tr_hi)]
        va = labelled[labelled["time_step"].between(va_lo, va_hi)]
        model = make_model().fit(tr[feature_cols], tr["label"].astype(int))
        p = model.predict_proba(va[feature_cols])[:, 1]
        rows.append({"split": name, "auc": roc_auc_score(va["label"], p),
                     "ap": average_precision_score(va["label"], p),
                     "n_val": len(va), "n_illicit": int(va["label"].sum())})
    res = pd.DataFrame(rows).set_index("split")
    folds = [s[0] for s in splits if s[0].startswith("fold")]
    res.loc["fold mean"] = [res.loc[folds, "auc"].mean(), res.loc[folds, "ap"].mean(),
                            np.nan, np.nan]
    return res

In [9]:
validation = {}
for name, cols in FEATURES.items():
    print(f"\nValidating {name} ({len(cols)} features) ...")
    validation[name] = validate(train_f, cols)
    print(validation[name].round(4).to_string())

summary = pd.DataFrame({name: res["auc"] for name, res in validation.items()}).T
print("\n=== Fold and horizon AUC by feature set ===")
print(summary.round(4).to_string())


Validating ppr_cmp (321 features) ...
              auc      ap    n_val  n_illicit
split                                        
fold1      0.9903  0.9760   3375.0      619.0
fold2      0.9631  0.8818   5311.0      566.0
fold3      0.9944  0.9866   2705.0      617.0
fold4      0.9984  0.9936   4330.0      690.0
horizon    0.9056  0.8146  11705.0     1773.0
fold mean  0.9865  0.9595      NaN        NaN

Validating prop34_cmp (639 features) ...
              auc      ap    n_val  n_illicit
split                                        
fold1      0.9914  0.9755   3375.0      619.0
fold2      0.9610  0.8794   5311.0      566.0
fold3      0.9948  0.9880   2705.0      617.0
fold4      0.9986  0.9941   4330.0      690.0
horizon    0.8812  0.8020  11705.0     1773.0
fold mean  0.9865  0.9593      NaN        NaN

Validating ppr_multi_cmp (639 features) ...
              auc      ap    n_val  n_illicit
split                                        
fold1      0.9913  0.9747   3375.0      619.0


## 5. Results and discussion

Development results from this validation scheme, with the leaderboard scores they
produced:

| Feature set | Fold mean AUC | Public LB |
|---|---|---|
| raw features only (159) | 0.9830 | 0.94351 |
| + component features | 0.9826 | 0.94864 |
| + 1/2-hop propagation | 0.9821 | 0.95670 |
| 2/3/4-hop propagation | 0.9865 | 0.95937 |
| PageRank, α = 0.15 | 0.9865 | 0.96034 |
| three decays | 0.9859 | 0.96057 |
| six decays | 0.9863 | 0.96066 |
| **rank blend of four models** | — | **0.96124** |

Three findings shaped the final choices.

**Information beyond one hop was the whole story.** Component features and multi-hop
propagation account for the entire rise from 0.9435 to 0.9612. Hand-built one-hop
aggregates (neighbour means, maxima, standard deviations) all failed, which fits the
data description: the supplied features already contain them.

**Validation separated good from bad models but could not rank close ones.** Across nine
probe submissions the rank correlation between validation score and public leaderboard
was 0.00. It correctly placed a GraphSAGE network far below the tree models (0.84
validation, 0.88 leaderboard), but it could not distinguish 0.955 from 0.961, and six
configurations that improved every validation metric went on to lose on test. Every
split ends at step 35 while the test set covers 36–49, a regime no split observes.

**Blending helped only between near-equal, mechanistically different models.** The four
blended here sit within 0.0007 of one another and summarise the neighbourhood
differently; blending gained each time it was tried under those conditions, and lost
whenever members differed by more than about 0.005 or shared a representation.

## 6. Final model and submission

Each model is retrained on **all** labelled training rows (steps 1–35, unknown labels
excluded) with exactly the validated settings, then combined by average rank — ranks
rather than probabilities, because AUC depends only on ordering and the models are
calibrated differently.

The weights mirror how the submission was assembled: `ppr_cmp` and `prop34_cmp` are
averaged equally; that pair is combined with `ppr_multi_cmp` in a 2:1 ratio; and the
result is combined with `ppr_fine_multi_cmp` in a 3:1 ratio.

In [10]:
def fit_predict_full(frame_tr, frame_te, feature_cols):
    labelled = frame_tr[frame_tr["label"].notna()]
    model = make_model().fit(labelled[feature_cols], labelled["label"].astype(int))
    return model.predict_proba(frame_te[feature_cols])[:, 1]


def rank01(p):
    return rankdata(p) / (len(p) + 1)


preds = {}
for name, cols in FEATURES.items():
    preds[name] = fit_predict_full(train_f, test_f, cols)
    print(f"{name:20s} mean {preds[name].mean():.4f}, median {np.median(preds[name]):.4f}")

pair = 0.5 * rank01(preds["ppr_cmp"]) + 0.5 * rank01(preds["prop34_cmp"])
with_multi = (2 / 3) * rank01(pair) + (1 / 3) * rank01(preds["ppr_multi_cmp"])
final = 0.75 * rank01(with_multi) + 0.25 * rank01(preds["ppr_fine_multi_cmp"])

print("\nRank correlation between the four models (lower = more diverse):")
print(pd.DataFrame({k: rank01(v) for k, v in preds.items()}).corr(method="spearman").round(4).to_string())

ppr_cmp              mean 0.0506, median 0.0001
prop34_cmp           mean 0.0498, median 0.0001
ppr_multi_cmp        mean 0.0499, median 0.0001
ppr_fine_multi_cmp   mean 0.0503, median 0.0001

Rank correlation between the four models (lower = more diverse):
                    ppr_cmp  prop34_cmp  ppr_multi_cmp  ppr_fine_multi_cmp
ppr_cmp              1.0000      0.9569         0.9769              0.9779
prop34_cmp           0.9569      1.0000         0.9618              0.9605
ppr_multi_cmp        0.9769      0.9618         1.0000              0.9815
ppr_fine_multi_cmp   0.9779      0.9605         0.9815              1.0000


In [11]:
# ── Write and validate the submission ─────────────────────────────────────────
submission = pd.DataFrame({"index": test["index"].to_numpy(), "target": final})

assert len(submission) == len(sample), "row count does not match sample_submission"
assert submission["index"].equals(sample["index"]), "index does not match sample_submission"
assert np.isfinite(submission["target"]).all(), "non-finite predictions"
assert submission["target"].between(0, 1).all(), "predictions outside [0, 1]"
assert list(submission.columns) == ["index", "target"], "unexpected columns"

path = OUT_DIR / "e1156454_submission.csv"
submission.to_csv(path, index=False)
print(f"All checks passed. Wrote {path} ({len(submission):,} rows)")
submission.head()

All checks passed. Wrote submissions/e1156454_submission.csv (15,329 rows)


,index,target
0,0,0.591031
1,1,0.721331
2,2,0.668509
3,3,0.529566
4,4,0.290941


## 7. Conclusion

The model is histogram gradient boosting with class weighting at its default settings,
so **none of the improvement came from the model**. Four hyperparameter searches (two
properly nested), stacking, bagging, pseudo-labelling, positive-unlabelled learning,
covariate-shift weighting, similarity-based label transfer, a GraphSAGE network and
three other boosting libraries were all tested and all scored lower.

What worked was representation. The supplied features already summarise each
transaction's immediate neighbourhood, so the remaining signal sat further out in the
payment graph — two to four hops away, or mixed across distances with a PageRank decay —
together with component structure. Public AUC rose from 0.94351 to 0.96124 on that basis
plus a rank blend of four such representations.

The clearest limitation is that chronological validation could not rank close
candidates, because no split can observe the period the model is scored on. Six
configurations improved every validation metric and still lost on test. In a deployed
anti-money-laundering system there is no leaderboard to fall back on, so closing that
gap would matter more than any further modelling: most plausibly by obtaining labels
from the period being scored, which is exactly what this setting withholds.

*Reproducibility note.* This notebook is a self-contained reimplementation of the staged
pipeline in `src/`. Validation figures agree with the development runs to about the
third decimal; the small differences come from how the component features treat nodes
with no edges. The models, settings and feature definitions are those of the 0.96124
submission.